![RAG](./images/image.png)


In [ ]:
!pip install langchain
!pip install -U langchain-community
!pip install sentence-transformers
!pip install faiss-gpu
!pip install pypdf
!pip install langchain_ollama


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for faiss-gpu



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


TXTs → split → embed → FAISS vector store

Ollama LLM wrapped by LangChain for generation

Conversational memory to keep context (window buffer)

RetrievalQA chain returns answer + source docs

In [16]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.memory import ConversationBufferMemory

# --- Step 1: Load and split documents ---
txt_paths = ["./resumes/resume1.txt", "./resumes/resume2.txt", "./resumes/resume3.txt"]
all_docs = []

for txt_path in txt_paths:
    loader = TextLoader(txt_path)
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    split_docs = text_splitter.split_documents(docs)
    for doc in split_docs:
        doc.metadata["source"] = txt_path
    all_docs.extend(split_docs)


In [17]:
# --- Step 2: Embeddings and vectorstore ---
embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cpu"}
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs=model_kwargs)
vectorstore = FAISS.from_documents(all_docs, embeddings)

vectorstore.save_local("faiss_index_")
persisted_vectorstore = FAISS.load_local("faiss_index_", embeddings, allow_dangerous_deserialization=True)
retriever = persisted_vectorstore.as_retriever(search_kwargs={"k": 5})

### Open a terminal and:
How to Install Ollama on Windows
Download the installer
Go to:
https://ollama.com/download
and click Windows to get the .exe installer.

Run the installer

It will install Ollama as a background service.

By default, Ollama will start automatically when Windows boots.

Open a Command Prompt or PowerShell
Then run: ollama pull llama3.1
This will download the LLaMA 3.1 model to your PC.

Start the server (optional)
Usually, Ollama on Windows auto-starts the service.
But if needed, run: ollama serve
It will listen on http://localhost:11434 for API requests.

Test it
Run: ollama run llama3.1
Then type a prompt and see if it responds.



In [18]:
# --- Step 3: Define prompt including chat history ---
prompt = PromptTemplate(
    template="""You are an assistant that helps Harry Potter fans by answering questions based on the content of the Harry Potter books provided as documents.
Use the following conversation history and documents to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
When possible, sprinkle in thematic words and references from the Harry Potter universe to make your answers catchy and engaging.

Chat History:
{chat_history}

Question: {question}
Documents:
{documents}

Answer:""",
    input_variables=["chat_history", "question", "documents"],
)


In [19]:
# --- Step 4: Setup LLM and chain with output parser ---
llm = ChatOllama(model="llama3.1", temperature=0)
rag_chain = prompt | llm | StrOutputParser()


In [20]:
# --- Step 5: Create memory instance ---
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=False)

In [21]:
# --- Step 6: Update RAGApplication to use memory ---
class RAGApplication:
    def __init__(self, retriever, rag_chain, memory):
        self.retriever = retriever
        self.rag_chain = rag_chain
        self.memory = memory

    def run(self, question: str) -> str:
        # Retrieve relevant docs
        documents = self.retriever.invoke(question)
        doc_texts = "\n".join([doc.page_content for doc in documents])
        
        # Get current chat history string from memory
        chat_history = self.memory.load_memory_variables({}).get("chat_history", "")
        
        # Prepare inputs for chain
        inputs = {
            "chat_history": chat_history,
            "question": question,
            "documents": doc_texts,
        }

        # Get answer
        answer = self.rag_chain.invoke(inputs)

        # Save interaction to memory (human question + AI answer)
        self.memory.save_context({"question": question}, {"answer": answer})

        return answer

In [22]:
# --- Step 7: Use RAG with memory ---
rag_application = RAGApplication(retriever, rag_chain, memory)

In [ ]:
# Example conversation
while True:
    user_question = input("\nAsk a question (or type 'exit' to quit): ")
    if user_question.lower() == "exit":
        break
    answer = rag_application.run(user_question)
    print("\nAnswer:", answer)



Answer: Hermione Granger is a brilliant and resourceful young witch who becomes one of Harry's closest friends at Hogwarts. She's known for her quick thinking and sharp mind, often helping Harry and Ron out of tricky situations with her clever ideas and magical prowess. As a Gryffindor student, Hermione proves herself to be a true Gryffindor spirit, brave and loyal in the face of danger.

Answer: Hermione Granger is indeed a close friend of Harry Potter's, and one of his most trusted companions throughout his adventures at Hogwarts. She's a brilliant and resourceful witch who often helps Harry out of tricky situations with her quick thinking and magical prowess. Together, they form an unbreakable bond, much like the ties that bind Gryffindor house together in the face of adversity.

Answer: The basilisk was killed by Harry Potter with the help of Fawkes, Dumbledore's loyal phoenix, who blinded the serpent and brought Harry the sword of Godric Gryffindor. With the sword, Harry was able